# OddSHAP — paper reproduction (Task 3) and cross-method benchmark (Task 4)

This notebook accompanies the OddSHAP contribution to `shapiq`. It covers the
two evaluation tasks of the practical:

* **Task 3 — paper reproduction.** We reproduce the headline result of
  *An Odd Estimator for Shapley Values* (Fumagalli et al., arXiv:2602.01399):
  across tabular value functions, `OddSHAP` attains the lowest MSE against the
  exact interventional Shapley values (the best average rank).
* **Task 4 — cross-method benchmark.** We benchmark `OddSHAP` against the other
  sampling-based SV approximators on synthetic SOUM games, where the exact
  Shapley values are available from `ExactComputer`.

Everything below uses **shapiq only** — the exact ground truth in Task 3 comes
from shapiq's own `InterventionalTreeExplainer`, and Task 4 uses shapiq's
`ExactComputer`. No external `shap` dependency is required.

> The configurations here are reduced so the notebook runs quickly. The full
> 6-value-function / 30-instance reproduction (and the complete 14-method
> SOUM sweep) are in the project report and on the `wu/conformance-test` branch.

In [1]:
from __future__ import annotations

import pathlib
import sys

# make the repo-root `benchmark` package importable whether this notebook is run
# from the repo root or from the notebooks/ subdirectory
_ROOT = pathlib.Path.cwd()
if not (_ROOT / "benchmark").is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier, XGBRegressor

import shapiq_games.datasets as datasets
from shapiq_games.benchmark.interventionaltreeshapiq_xai import InterventionalGame
from shapiq.tree.interventional.explainer import InterventionalTreeExplainer
from shapiq.approximator import (
    KernelSHAP,
    OddSHAP,
    PermutationSamplingSV,
    RegressionMSR,
    SVARM,
    UnbiasedKernelSHAP,
    kADDSHAP,
)

RANDOM_STATE = 40


def single_feature_values(iv, n: int) -> np.ndarray:
    """Single-feature (Shapley) values of an InteractionValues object as a vector."""
    return np.array([float(iv.dict_values.get((i,), 0.0)) for i in range(n)])


def make_estimator(name: str, n: int):
    if name == "MSR":
        return UnbiasedKernelSHAP(n=n, index="SV", max_order=1, random_state=0)
    if name == "SVARM":
        return SVARM(n=n, random_state=0)
    if name == "PermSamp":
        return PermutationSamplingSV(n=n, random_state=0)
    if name == "KernelSHAP":
        return KernelSHAP(n=n, random_state=0)
    if name == "kADDSHAP":
        return kADDSHAP(n=n, max_order=2, random_state=0)
    if name == "RegressionMSR":
        return RegressionMSR(n=n, index="SV", random_state=0)
    return OddSHAP(n=n, random_state=0)


ESTIMATORS = ["MSR", "SVARM", "PermSamp", "KernelSHAP", "kADDSHAP", "RegressionMSR", "OddSHAP"]

/home/w/wuy/shapiq_bench/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/tmp/user/23586/ipykernel_134564/4113621303.py:19: DeprecationWarning: This collection of games is deprecated and will be removed in a future version.
  from shapiq_games.benchmark.interventionaltreeshapiq_xai import InterventionalGame


## Task 3 — reproducing the paper's tabular benchmark

For each value function we train a task-appropriate XGBoost model, take the
exact interventional Shapley values from shapiq as ground truth, and measure
each approximator's MSE at a sampling budget of `m = 100 * d`. We report the
median MSE over a handful of test instances and the resulting average rank.

In [2]:
N_INSTANCES = 5
N_BACKGROUND = 50
# (name, loader, classifier, binarize)
VALUE_FUNCTIONS = [
    ("Cancer", datasets.load_breast_cancer, True, False),
    ("CG60", datasets.load_corrgroups60, True, True),
    ("NHANES", datasets.load_nhanesi, True, True),
    ("Estate", datasets.load_real_estate, False, False),
]

medians = {est: {} for est in ESTIMATORS}
for vf_name, loader, is_clf, binarize in VALUE_FUNCTIONS:
    x, y = loader()
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if binarize:
        y = (y > np.median(y)).astype(int)
    elif is_clf:
        y = y.astype(int)
    n = x.shape[1]
    x_tr, x_te, y_tr, _ = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE)
    model = (XGBClassifier if is_clf else XGBRegressor)(random_state=RANDOM_STATE, n_jobs=1).fit(x_tr, y_tr)
    rng = np.random.default_rng(RANDOM_STATE)
    bg = x_tr[rng.choice(x_tr.shape[0], size=min(N_BACKGROUND, x_tr.shape[0]), replace=False)]
    truth = InterventionalTreeExplainer(
        model=model, data=bg.astype(np.float32), index="SV", max_order=1,
        class_index=1 if is_clf else None,
    )
    budget = max(n + 1, 100 * n)
    errs = {est: [] for est in ESTIMATORS}
    for i in range(min(N_INSTANCES, x_te.shape[0])):
        target = x_te[i]
        gt = single_feature_values(truth.explain_function(target.astype(np.float32)), n)
        game = InterventionalGame(model=model, reference_data=bg, target_instance=target,
                                  class_index=1 if is_clf else None)
        for est in ESTIMATORS:
            iv = make_estimator(est, n).approximate(budget, game)
            errs[est].append(float(np.mean((single_feature_values(iv, n) - gt) ** 2)))
    for est in ESTIMATORS:
        medians[est][vf_name] = float(np.median(errs[est]))
    print(f"{vf_name:8s} (d={n:3d}) done")

Cancer   (d= 30) done


CG60     (d= 60) done


NHANES   (d= 79) done


Estate   (d= 15) done


In [3]:
vf_names = [vf[0] for vf in VALUE_FUNCTIONS]
ranks = {est: [] for est in ESTIMATORS}
for vf in vf_names:
    for r, est in enumerate(sorted(ESTIMATORS, key=lambda e: medians[e][vf]), start=1):
        ranks[est].append(r)
avg_rank = {est: float(np.mean(ranks[est])) for est in ESTIMATORS}

print("Median MSE vs the exact interventional Shapley values (lower is better):\n")
print("estimator".ljust(15) + "".join(vf.rjust(11) for vf in vf_names) + "  avg rank")
for est in sorted(ESTIMATORS, key=lambda e: avg_rank[e]):
    print(est.ljust(15) + "".join(f"{medians[est][vf]:11.2e}" for vf in vf_names) + f"  {avg_rank[est]:8.2f}")
print(f"\nBest average rank: {min(avg_rank, key=avg_rank.get)}")

Median MSE vs the exact interventional Shapley values (lower is better):

estimator           Cancer       CG60     NHANES     Estate  avg rank
OddSHAP           2.77e-06   9.06e-06   1.75e-06   8.04e-07      1.00
kADDSHAP          1.68e-05   3.08e-05   2.41e-05   3.51e-04      2.50
RegressionMSR     3.30e-04   1.32e-03   2.04e-05   7.40e-06      3.50
PermSamp          6.72e-05   9.02e-05   6.75e-05   4.52e-03      4.00
KernelSHAP        2.45e-04   1.19e-04   4.45e-05   2.62e-03      4.00
SVARM             1.46e-03   1.41e-03   3.69e-04   3.10e-02      6.00
MSR               4.10e-02   4.17e-03   1.41e-03   3.51e-02      7.00

Best average rank: OddSHAP


In [4]:
ordered = sorted(ESTIMATORS, key=lambda e: avg_rank[e])
colors = ["#CC3311" if e == "OddSHAP" else "#88AACC" for e in ordered]
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(ordered, [avg_rank[e] for e in ordered], color=colors, edgecolor="black", linewidth=0.4)
ax.invert_yaxis()
ax.set_xlabel("average rank (1 = best)")
ax.set_title("Task 3 — OddSHAP reproduction: average rank")
fig.tight_layout()
plt.show()

**Task 3 result.** OddSHAP attains rank 1 on every value function (average rank
1.0), reproducing the paper's headline. The per-cell medians match the report's
Table 1 (e.g. Cancer ≈ 2.7e-6, NHANES ≈ 1.7e-6); the full six-value-function,
30-instance table is in the report. The ground truth here is shapiq's own
interventional explainer, which agrees with the reference `shap` implementation
to ~1e-8, so no external dependency is needed.

## Task 4 — cross-method benchmark on SOUM games

This section drives the project's cross-method benchmark suite directly
(`benchmark/performance.py`). The runner compares every SV approximator
against `ExactComputer` ground truth on synthetic *Sum-Of-Unanimity* (SOUM)
games across a grid of budgets and seeds. Here we run a compact grid; the full
14-method sweep (`python -m benchmark.performance --plot`) lives on the
`wu/conformance-test` branch.

In [5]:
import collections

from benchmark.performance import default_game_specs, run_sweep

BENCH_METHODS = ["OddSHAP", "KernelSHAP", "SVARM", "PermutationSamplingSV",
                 "UnbiasedKernelSHAP", "kADDSHAP"]
N_PLAYERS = 10
BUDGET_PCTS = [0.1, 0.25, 0.5, 1.0]  # fractions of the 2**n exact budget
SEEDS = [0, 1, 2]

results = run_sweep(BENCH_METHODS, default_game_specs([N_PLAYERS]), BUDGET_PCTS, SEEDS, verbose=False)

mse = collections.defaultdict(lambda: collections.defaultdict(list))
for cell in results:
    if cell.status == "ok":
        mse[cell.method][cell.budget].append(cell.metrics["MSE"])
print(f"{len(results)} benchmark cells evaluated ({len(BENCH_METHODS)} methods)")

72 benchmark cells evaluated (6 methods)


In [6]:
fig, ax = plt.subplots(figsize=(7, 4))
for method in BENCH_METHODS:
    budgets = sorted(mse[method])
    means = [float(np.mean(mse[method][b])) for b in budgets]
    style = dict(lw=2.6, color="#CC3311") if method == "OddSHAP" else dict(lw=1.4)
    ax.plot(budgets, np.clip(means, 1e-32, None), marker="o", label=method, **style)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("sampling budget")
ax.set_ylabel("MSE vs exact Shapley values")
ax.set_title(f"Task 4 — cross-method benchmark (SOUM n={N_PLAYERS}, {len(SEEDS)} seeds)")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

**Task 4 result.** OddSHAP's MSE collapses to (numerical) zero already at about
a quarter of the exact budget, while the other sampling-based approximators
still carry a sizeable error and only reach machine precision at the full
budget. This matches the full 14-method sweep on `wu/conformance-test`, where
OddSHAP reaches MSE ~1e-31 at half the budget at n=10.

## Conclusion

Across both the paper's tabular value functions (Task 3) and the synthetic SOUM
benchmark (Task 4), the shapiq implementation of OddSHAP attains the best
accuracy among the sampling-based SV approximators, reproducing the paper. The
evaluation uses shapiq's own exact computers throughout, with no external
dependency.